# 07. Correlación: ¿qué tanto se parecen dos señales?

**Objetivos de aprendizaje.** Al terminar podrás:

1. Calcular la **correlación cruzada** y la **autocorrelación** a mano y con código.
2. Explicar la diferencia (y la relación) entre **correlación** y **convolución**.
3. Usar la correlación para **estimar un retardo** (radar/sonar).
4. Usar la autocorrelación para encontrar el **periodo de una señal con ruido**.

**Requisito:** notebooks 04 y 06.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. La idea

Para medir qué tanto se parecen dos señales, las **multiplicamos muestra a muestra y sumamos**:
- si son parecidas, los productos son casi todos positivos → la suma es **grande**;
- si no se parecen, los productos tienen signos al azar → la suma es **pequeña**.

Pero una señal puede estar **desplazada** respecto a la otra. Entonces lo hacemos para **cada desplazamiento** $l$ (*lag*):

$$r_{xy}[l] = \sum_{n} x[n]\, y[n-l]$$

$r_{xy}[l]$ es grande cuando $y$, desplazada $l$ muestras a la derecha, se parece a $x$.

### Paso a paso con $x = \{1, 2, 3\}$, $y = \{1, 1\}$ (ambas desde $n=0$)

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([1.0, 1.0])

# Los desplazamientos l en los que las señales se "tocan": de -(Ly-1) a Lx-1
lags = np.arange(-(len(y) - 1), len(x))
print("lags =", lags)

for l in lags:
    terminos = []
    suma = 0.0
    for n in range(len(x)):
        j = n - l                           # posición de y que se compara con x[n]
        if 0 <= j < len(y):
            terminos.append(f"x[{n}]·y[{j}]")
            suma = suma + x[n] * y[j]
    print(f"r[{l:2d}] = " + " + ".join(terminos) + f" = {suma}")

In [ ]:
import inspect
print(inspect.getsource(pds.correlacion_cruzada))

In [ ]:
lags, r = pds.correlacion_cruzada(x, y)
print("lags =", lags, "  r =", r)

# numpy: np.correlate(x, y, "full") calcula lo mismo (para el mismo rango de lags)
pds.comprobar_iguales(r, np.correlate(x, y, "full"), "coincide con np.correlate")

## 2. Correlación vs convolución

| | Fórmula | ¿Se invierte la segunda señal? |
|---|---|---|
| Convolución | $\sum_k x[k]\,h[n-k]$ | **Sí** (invertir y desplazar) |
| Correlación | $\sum_n x[n]\,y[n-l]$ | **No** (sólo desplazar) |

Por eso: $\;r_{xy}[l] = x[l] * y[-l]$ (correlacionar = convolucionar con la señal **invertida**).

In [ ]:
x = np.random.randn(8)
y = np.random.randn(5)
lags, r = pds.correlacion_cruzada(x, y)
r_por_convolucion = np.convolve(x, y[::-1])          # y[::-1] es y[-n]
pds.comprobar_iguales(r, r_por_convolucion, "correlación = convolución con y invertida")

## 3. Autocorrelación

Correlación de una señal **consigo misma**: $\;r_{xx}[l] = \sum_n x[n]\,x[n-l]$

**Propiedades** (las vamos a comprobar con código):
1. Es **simétrica**: $r_{xx}[-l] = r_{xx}[l]$.
2. Su **máximo** está en $l = 0$, y vale la **energía**: $r_{xx}[0] = \sum x[n]^2 = E$.
3. Si $x$ es **periódica** con periodo $N$, $r_{xx}$ tiene picos en $l = N, 2N, 3N, \dots$

In [ ]:
x = np.random.randn(50)
lags, r = pds.autocorrelacion(x)

pds.comprobar_iguales(r, r[::-1], "1) simétrica")
pds.comprobar(np.argmax(r) == np.where(lags == 0)[0][0], "2a) máximo en l = 0")
pds.comprobar_iguales(r[lags == 0][0], pds.energia(x), "2b) r[0] = energía")

pds.graficar_discreta(lags, r, titulo="Autocorrelación de ruido: sólo un pico en l = 0",
                      etiqueta_x="l (lag)", etiqueta_y="r_xx[l]")
plt.show()

> Nota: El **ruido blanco** no se parece a sí mismo desplazado: su autocorrelación es (casi) un impulso.
> Una señal **periódica** sí se parece a sí misma desplazada un periodo. Esa diferencia es la que nos permite encontrar periodos aun con ruido.

## 4. Aplicación 1: periodo de una señal con ruido

In [ ]:
n = np.arange(0, 300)
limpia = np.cos(2 * np.pi * n / 25)
ruidosa = limpia + 1.0 * np.random.randn(len(n))

print("Búsqueda directa (notebook 04):", pds.estimar_periodo(ruidosa), " <- falla con ruido")

lags, r = pds.autocorrelacion(ruidosa)
figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_continua(n, ruidosa, titulo="Senoidal de periodo 25 + mucho ruido", etiqueta_x="n", ax=ejes[0])
pds.graficar_continua(lags, r, titulo="Autocorrelación: picos cada 25 muestras", etiqueta_x="l", ax=ejes[1])
for k in range(-275, 300, 25):
    ejes[1].axvline(k, color="r", alpha=0.2)
plt.tight_layout()
plt.show()

**Cómo leer el periodo:** ignoramos el pico de $l=0$ (siempre es el máximo) y buscamos el **siguiente pico más alto** para $l > 0$.

In [ ]:
# Paso 1: quedarnos sólo con los lags positivos, a partir de un mínimo (para saltar el pico de l = 0)
l_minimo = 5
seleccion = lags >= l_minimo
lags_pos = lags[seleccion]
r_pos = r[seleccion]

# Paso 2: el lag con la correlación más alta
periodo = lags_pos[np.argmax(r_pos)]
print("Periodo estimado:", periodo, "muestras")

## 5. Aplicación 2: estimar un retardo (radar / sonar)

Un sonar emite un pulso $x[n]$; el eco regresa **atenuado, retrasado $D$ muestras y con ruido**:
$$y[n] = a\,x[n-D] + \text{ruido}[n]$$
La correlación $r_{yx}[l]$ es máxima en $l = D$. Con $D$ calculamos la distancia: $d = \dfrac{c \cdot (D/f_s)}{2}$
(se divide entre 2 porque el sonido va y regresa).

In [ ]:
fs = 48000                     # muestras/s
c = 343                        # velocidad del sonido en el aire, m/s

# Pulso emitido: 2 ms de una senoidal de 5 kHz (un "chirrido" corto)
n_p, t_p, pulso = pds.senoidal_digital(5000, fs, 0.002)

# Señal recibida: 50 ms de ruido, con el eco escondido en D = 1000 muestras
D_real = 1000
recibida = 0.5 * np.random.randn(pds.indice_de_muestra(0.05, fs))
recibida[D_real:D_real + len(pulso)] += 0.3 * pulso

D = pds.estimar_retardo(pulso, recibida)
distancia = c * (D / fs) / 2
print(f"Retardo estimado: D = {D} muestras = {D/fs*1000:.2f} ms  ->  distancia = {distancia:.2f} m")

lags, r = pds.correlacion_cruzada(recibida, pulso)
figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_continua(np.arange(len(recibida)), recibida, titulo="Señal recibida: ¿ves el eco?", etiqueta_x="n", ax=ejes[0])
pds.graficar_continua(lags, r, titulo="Correlación con el pulso: el eco aparece como un pico claro", etiqueta_x="l", ax=ejes[1])
plt.tight_layout()
plt.show()

## 6. Correlación normalizada: un "porcentaje de parecido"

Dividiendo entre $\sqrt{E_x E_y}$ el resultado queda entre $-1$ y $1$:
$$\rho_{xy}[l] = \frac{r_{xy}[l]}{\sqrt{E_x\,E_y}}$$

In [ ]:
n, x = pds.senoidal_digital_muestras(100, 0.05)
lags, rho_igual = pds.correlacion_normalizada(x, x)
lags, rho_neg = pds.correlacion_normalizada(x, -x)
lags, rho_ruido = pds.correlacion_normalizada(x, np.random.randn(100))
print("x con x  : ρ[0] =", round(rho_igual[lags == 0][0], 3))
print("x con -x : ρ[0] =", round(rho_neg[lags == 0][0], 3))
print("x con ruido: ρ[0] =", round(rho_ruido[lags == 0][0], 3))

## Comprueba tu comprensión

1. ¿En qué se diferencia la fórmula de la correlación de la de la convolución?
2. ¿Por qué el máximo de la autocorrelación siempre está en $l=0$?
3. Un sonar con $f_s = 10$ kHz detecta el eco con $D = 580$ muestras. ¿A qué distancia está el objeto? ($c = 343$ m/s)

<details><summary>Ver respuestas</summary>

1. En la correlación la segunda señal **no** se invierte.
2. Porque ninguna versión desplazada se parece más a la señal que ella misma (desigualdad de Cauchy-Schwarz).
3. $t = 0.058$ s → $d = 343 \times 0.058 / 2 \approx 9.95$ m.
</details>